#### Secrets

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

Our sample phrases cover three topics: freedom, friendship, and food.

In [2]:
phrases = [
    # Freedom
    "Freedom consists not in doing what we like, but in having the right to do what we ought.",
    "Those who deny freedom to others deserve it not for themselves.",
    "Liberty, when it begins to take root, is a plant of rapid growth.",
    "Freedom lies in being bold.",
    "Is freedom anything else than the right to live as we wish?",
    "I am no bird and no net ensnares me: I am a free human being with an independent will.",
    "The secret to happiness is freedom... And the secret to freedom is courage."
    "Freedom is the oxygen of the soul.", 
    "Life without liberty is like a body without spirit."
    # Friendship
    "There is nothing on this earth more to be prized than true friendship.",
    "There are no strangers here; Only friends you haven’t yet met.",
    "Friendship is the only cement that will ever hold the world together.",
    "A true friend is someone who is there for you when he'd rather be anywhere else.",
    "Friendship is the golden thread that ties the heart of all the world.", 
    "Your friend is the man who knows all about you and still likes you.",
    "A single rose can be my garden... a single friend, my world."
    # Food
    "One cannot think well, love well, sleep well, if one has not dined well.",
    "Let food be thy medicine and medicine be thy food.",
    "People who love to eat are always the best people.",
    "The only way to get rid of a temptation is to yield to it.",
    "Food is our common ground, a universal experience.",
    "Life is uncertain. Eat dessert first.",
    "All you need is love. But a little chocolate now and then doesn't hurt."
]

We have 20 phrases in total:

In [3]:
len(phrases)

20

In [4]:
ids = [f"id{i}" for i in range(len(phrases))]

# Vector DB

We can use a specialized database to store our embeddings, relate them to documents, and efficiently perform computations like cosine similarity.


The document database that we will use for our experiments is Chroma DB, a simple implementation of Vector DB that is commonly used for prototyping. 

A few useful references are: 
- [ChromaDB Documentation](https://docs.trychroma.com/docs/overview/introduction).
- [ChromaDB Cookbook](https://cookbook.chromadb.dev/running/running-chroma/#chroma-cli).

Chroma can be run locally in memory, locally using file persistence, or using a Docker container.

## Running Chroma Locally in Memory

Run Chroma DB in memory with persistence.

In [5]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./02_5_chromadb")

In [6]:
chroma_client.heartbeat()

1762831373312614900

### Embedding phrases into chromadb & 
### Performing a Search Using Embedding Function

Alternatively, we can define the embedding function at the moment in which we create the collection.

We can now re-use the collection name using an OpenAI embedding function. Notice that we pass the `api_key` parameter explicitly, as the environment variable name that holds the API key for Chroma DB and for the OpenAI library are different.

In [7]:
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

collection = chroma_client.create_collection(
    name = "nice_phrases",
    embedding_function = OpenAIEmbeddingFunction(
        api_key = os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small")
)
collection.add(documents = phrases, 
               ids = ids)

With the embedding function, we can now perform the query:

In [8]:
collection.query(
    query_texts = ["What is a friend?", "What is good food?"], 
    n_results = 2
)

{'ids': [['id10', 'id12'], ['id17', 'id15']],
 'embeddings': None,
 'documents': [["A true friend is someone who is there for you when he'd rather be anywhere else.",
   'Your friend is the man who knows all about you and still likes you.'],
  ['Food is our common ground, a universal experience.',
   'People who love to eat are always the best people.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None], [None, None]],
 'distances': [[0.9254988431930542, 1.0471810102462769],
  [1.0074349641799927, 1.1379879713058472]]}

In [9]:
#chroma_client.reset()          # AuthorizationError: Reset is disabled by config